## Imports

In [1]:
%load_ext autoreload
%autoreload 2

# Internal import
import deep_learning_land_use_classification.config as dataset
import deep_learning_land_use_classification.evaluation as evaluation

import torch
from transformers import AutoImageProcessor, AutoModel

from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
import wandb
import torch.nn as nn

/Users/user/miniforge3/envs/DL_landuse/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Wandb initialization for experiment tracking

run = wandb.init(
    entity="sstaheli52-wageningen-university-and-research",
    project="DL_land-use-classification",
    dir=dataset.WANDB_DIR,
    config={
        "architecture": "dinov3-vitl16",
        "model_name": "facebook/dinov3-vitl16-pretrain-sat493m",
        "pretrained": True,
        "pretraining_dataset": "sat493m",
        "pretraining_source": "huggingface",
        "weights": "facebook/dinov3-vitl16-pretrain-sat493m",
        "epochs": 5,
        "batch_size": 32,
        "learning_rate": 1e-4,
        "optimizer": "Adam",
        "loss": "BCEWithLogitsLoss",
        "num_classes": dataset.num_classes,
    },
)

config = run.config

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/user/.netrc.
wandb: Currently logged in as: sstaheli52 (sstaheli52-wageningen-university-and-research) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
pretrained_model_name = "facebook/dinov3-vitl16-pretrain-sat493m"
processor = AutoImageProcessor.from_pretrained(pretrained_model_name)

In [7]:
class MultiLabelDataset(Dataset):
    def __init__(self, dataframe, class_names, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.class_names = class_names
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["path"]).convert("RGB")

        inputs = processor(images=image, return_tensors="pt") # converts image to tensor and normalizes it
        inputs = {k: v.squeeze(0) for k, v in inputs.items()} # remove batch dimension

        label = torch.tensor(row[self.class_names].values.astype(np.float32))

        return inputs, label

In [8]:
# Create datasets and dataloaders
train_dataset = MultiLabelDataset(dataset.train_df, dataset.class_names, processor)
test_dataset  = MultiLabelDataset(dataset.test_df, dataset.class_names, processor)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)


In [9]:
# this is a simple wrapper around the DINO backbone to add a classification head on top
class DinoClassifier(torch.nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        self.classifier = torch.nn.Linear(
            backbone.config.hidden_size,
            num_classes
        )

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values) # get the output of the backbone
        features = outputs.pooler_output
        return self.classifier(features)

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ("Using device:", device)

backbone = AutoModel.from_pretrained(pretrained_model_name)

model = DinoClassifier(backbone, num_classes=config.num_classes).to(device)
model = model.to(device)


Using device: cpu


Loading weights: 100%|██████████| 415/415 [00:00<00:00, 6797.90it/s]


In [ ]:
labels = dataset.train_df[dataset.class_names].values

# Count positives per class
pos_counts = labels.sum(axis=0)
neg_counts = len(labels) - pos_counts

# Avoid division by zero
pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-5), dtype=torch.float32)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.01)

In [ ]:
run.watch(model, log="all", log_freq=100)

### Train Model

In [23]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for inputs, labels in loader:
        pixel_values = inputs["pixel_values"].to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(pixel_values)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x16a3f25f0>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 16c459870, raw_cell="def train(model, loader, optimizer, criterion):
  .." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Users/user/Dropbox/My_Mac/Documents/Wageningen/Coursework/Period_5/Deep_learning/DL_landuse_classification_project/Deep-learning-land-use-classification/notebooks/3.01_multi_label_ViT.ipynb#X24sZmlsZQ%3D%3D>,),kwargs {}:


ConnectionResetError: Connection lost

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x16a3f25f0>> (for post_run_cell), with arguments args (<ExecutionResult object at 16c459d50, execution_count=23 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 16c459870, raw_cell="def train(model, loader, optimizer, criterion):
  .." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Users/user/Dropbox/My_Mac/Documents/Wageningen/Coursework/Period_5/Deep_learning/DL_landuse_classification_project/Deep-learning-land-use-classification/notebooks/3.01_multi_label_ViT.ipynb#X24sZmlsZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

In [24]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for inputs, labels in loader:
            pixel_values = inputs["pixel_values"].to(device)
            labels = labels.to(device)

            outputs = model(pixel_values)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

    return total_loss / len(loader)

Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x16a3f25f0>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 158030c70, raw_cell="def evaluate(model, loader, criterion):
    model..." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Users/user/Dropbox/My_Mac/Documents/Wageningen/Coursework/Period_5/Deep_learning/DL_landuse_classification_project/Deep-learning-land-use-classification/notebooks/3.01_multi_label_ViT.ipynb#X30sZmlsZQ%3D%3D>,),kwargs {}:


ConnectionResetError: Connection lost

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x16a3f25f0>> (for post_run_cell), with arguments args (<ExecutionResult object at 1580303a0, execution_count=24 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 158030c70, raw_cell="def evaluate(model, loader, criterion):
    model..." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Users/user/Dropbox/My_Mac/Documents/Wageningen/Coursework/Period_5/Deep_learning/DL_landuse_classification_project/Deep-learning-land-use-classification/notebooks/3.01_multi_label_ViT.ipynb#X30sZmlsZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost